In [5]:
from SPARQLWrapper import SPARQLWrapper, JSON, CSV
import subprocess
import time
import os
import requests
import rdflib
import pandas as pd
import numpy as np
import networkx as nx
import itertools
import glob
import plotly.express as px
from utility import *

In [6]:
current_directory = os.getcwd()
BioPAX_Ontology_file_path = os.path.join(current_directory, '../../Data/BioPAX/BioPAXOntology/biopax-level3.owl')
ReactomeBioPAX_file_path = os.path.join(current_directory, '../../Data/BioPAX/ReactomeBioPAX/Homo_sapiens_v96.owl')
tgf_beta_pathway_path = os.path.join(current_directory, '../../Data/BioPAX/ReactomeBioPAX/R-HSA-9006936.xml')
signaling_tgf_beta_complex = os.path.join(current_directory, '../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml')
PathwayAbstractionsFolder = os.path.join(current_directory, '../../Results/PathwayAbstraction')

prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
"""
endpoint = "http://localhost:3030/tgf_beta"

### comembership clique

In [8]:
prefixes = """
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#>
"""

query="""
SELECT DISTINCT ?id1 ?id2
WHERE {
    VALUES ?db1 { "UniProt" "UniProt Isoform" }
    ?entityRef1 rdf:type/(rdfs:subClassOf*) bp3:ProteinReference .
    ?entityRef1 bp3:xref ?entityRef1Xref .
    ?entityRefXref1 rdf:type bp3:UnificationXref .
    ?entityRefXref1 bp3:db ?db1 .
    ?entityRefXref1 bp3:id ?id1 .
    
    VALUES ?db2 { "UniProt" "UniProt Isoform" }
    ?entityRef2 rdf:type/(rdfs:subClassOf*) bp3:ProteinReference .
    ?entityRef2 bp3:xref ?entityRef2Xref .
    ?entityRef2Xref rdf:type bp3:UnificationXref .
    ?entityRef2Xref bp3:db ?db2 .
    ?entityRef2Xref bp3:id ?id2 .
    FILTER (?id1 < ?id2)
}
"""

command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', signaling_tgf_beta_complex,
    '--file', BioPAX_Ontology_file_path,
    '/tgf_beta'
]
process = subprocess.Popen(command)
time.sleep(30)
sparql = SPARQLWrapper(endpoint)
sparql.setQuery(prefixes + query)
sparql.setReturnFormat(CSV)
try:
    results = sparql.query().convert()
    output_filename = os.path.join(f"../../Results/PathwayComembership/TGFbeta/R-HSA-170834_ComembershipClique.csv")
    with open(output_filename, 'wb') as f:
        f.write(results)
    print(f"Results saved in {output_filename}")
except Exception as e:
    print(f"Error in SPARQL query: {e}")
process.kill()
time.sleep(30)

10:56:46 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml
10:56:46 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl
10:56:46 INFO  Server          :: Running in read-only mode for /tgf_beta
10:56:46 INFO  Server          :: Apache Jena Fuseki 4.9.0
10:56:46 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
10:56:46 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/run
10:56:46 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/run/shiro.ini
10:56:46 INFO 

Results saved in ../../Results/PathwayComembership/TGFbeta/R-HSA-170834_ComembershipClique.csv


### weight comembership clique

In [10]:
query = """
SELECT (COUNT(DISTINCT ?id) AS ?nbID)
WHERE {
    VALUES ?db { "UniProt" "UniProt Isoform" }
    ?entityRef rdf:type/(rdfs:subClassOf*) bp3:ProteinReference .
    ?entityRef bp3:xref ?entityRefXref .
    ?entityRefXref bp3:db ?db .
    ?entityRefXref bp3:id ?id .
}
"""

# ------------------------------------------------------------------
# 0 - Load pathway abstraction
# ------------------------------------------------------------------
pathway_abstraction = pd.read_csv(
    "../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_WeightedPathwayAbstraction.csv",
    sep=",", header=0
)
G = create_networkx_graph(pathway_abstraction)
hierarchy_graph = extract_subgraph_by_interaction(G, "abstraction:IsAComponentOf")

# ------------------------------------------------------------------
# 1 - get number of Uniprot IDs per pathway
# ------------------------------------------------------------------
command = [
    '/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0/fuseki-server',
    '--file', signaling_tgf_beta_complex,
    '--file', BioPAX_Ontology_file_path,
    '/tgf_beta'
]
process = subprocess.Popen(command)
time.sleep(30)

try:
    sparql = SPARQLWrapper(endpoint)
    sparql.setQuery(prefixes + query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    nbEr_total = int(results["results"]["bindings"][0]["nbID"]["value"])

    # ------------------------------------------------------------------
    # 2 - get list of Uniprot IDs per pathway
    # ------------------------------------------------------------------
    er_per_pathway = pd.read_csv(
        f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_UpPerPathway.csv",
        sep=",", header=0
    )

    dico_er_per_pathway = {}
    for pathway, id_ in er_per_pathway.iloc[:, [0, 1]].values:
        dico_er_per_pathway.setdefault(pathway, []).append(id_)

    # ------------------------------------------------------------------
    # 3 - compute Score IC (with content of UniProt IDs) of pathways
    # ------------------------------------------------------------------
    dict_score_ic_pathways = {
        pathway: compute_ic_er_pathway(pathway, dico_er_per_pathway, nbEr_total)
        for pathway in dico_er_per_pathway
    }

    score_IC_pathways = pd.DataFrame({
        'pathway': list(dict_score_ic_pathways.keys()),
        'score': list(dict_score_ic_pathways.values())
    })
    score_IC_pathways.to_csv(
        f"../../Results/PathwayComembership/TGFbeta/R-HSA-170834_ScoreICPathways.csv",
        sep=",", header=True, index=False
    )

    # ------------------------------------------------------------------
    # 4 - get next step pathway scores (dict lookup insR-HSA-170834tead of DataFrame .loc)
    # ------------------------------------------------------------------
    list_pathways = []
    seen_pathways = set()
    for pathway1, _, pathway2 in pathway_abstraction.iloc[:, [0, 1, 2]].values:
        if pathway1 not in seen_pathways:
            seen_pathways.add(pathway1)
            list_pathways.append(pathway1)
        if pathway2 not in seen_pathways:
            seen_pathways.add(pathway2)
            list_pathways.append(pathway2)

    mat_lookup = {}
    for pathway1, interaction, pathway2, weight_er in pathway_abstraction.iloc[:, [0, 1, 2, 3]].values:
        if interaction == "abstraction:NextStepPathway":
            mat_lookup[(pathway1, pathway2)] = weight_er
            mat_lookup[(pathway2, pathway1)] = weight_er

    # ------------------------------------------------------------------
    # 5 - get pathway ancestors
    # ------------------------------------------------------------------
    dico_ancestors = {}
    for pathway in list_pathways:
        anc, _ = find_ancestors_and_descendants(hierarchy_graph, pathway)
        dico_ancestors[pathway] = anc

    up_per_pathway = pd.read_csv(
        f"../../Results/PathwayAbstraction/TGFbeta/R-HSA-170834_UpPerPathway.csv",
        sep=",", header=0
    )

    dico_parents_up = {}
    for pathway_val, prot_val in up_per_pathway.iloc[:, [0, 1]].values:
        dico_parents_up.setdefault(prot_val, []).append(pathway_val)

    root = find_root(G, "abstraction:IsAComponentOf")

    # precompute shortest paths
    reversed_hierarchy = hierarchy_graph.reverse()
    shortest_path_len_from_root = {}
    for pathway in list_pathways:
        try:
            shortest_path_len_from_root[pathway] = len(
                nx.shortest_path(reversed_hierarchy, root, pathway)
            )
        except nx.NetworkXNoPath:
            shortest_path_len_from_root[pathway] = 0

    dico_most_precise_parent = {}
    print(len(dico_parents_up))
    for prot, parents in dico_parents_up.items():
        if len(parents) == 1:
            dico_most_precise_parent[prot] = parents
        else:
            shortest_paths = {
                parent: shortest_path_len_from_root.get(parent, 0)
                for parent in parents
            }
            max_len = max(shortest_paths.values())
            dico_most_precise_parent[prot] = [
                key for key, val in shortest_paths.items() if val == max_len
            ]

    # ------------------------------------------------------------------
    # 6 - weight comembership clique
    # ------------------------------------------------------------------
    comembership_clique = pd.read_csv(
        f"../../Results/PathwayComembership/TGFbeta/R-HSA-170834_ComembershipClique.csv",
        sep=",", header=0
    )
    ancestors_cache = {}
    mica_cache = {}
    next_step_cache = {}

    def get_ancestors_for_entity(entity):
        cached = ancestors_cache.get(entity)
        if cached is None:
            parents = dico_most_precise_parent[entity]
            ancestors_pathways = get_pathways_and_ancestor(parents, dico_ancestors)
            cached = (parents, ancestors_pathways)
            ancestors_cache[entity] = cached
        return cached

    def get_mica(parents1, parents2):
        key = (tuple(sorted(parents1)), tuple(sorted(parents2)))
        cached = mica_cache.get(key)
        if cached is None:
            cached = find_mica_from_several_pathways(
                hierarchy_graph, root, parents1, parents2,
                dico_er_per_pathway, nbEr_total
            )
            mica_cache[key] = cached
        return cached

    def get_max_next_step_score(ancestors1, ancestors2):
        key = (tuple(sorted(ancestors1)), tuple(sorted(ancestors2)))
        cached = next_step_cache.get(key)
        if cached is None:
            best = 0
            for p1, p2 in itertools.product(ancestors1, ancestors2):
                if p1 != p2:
                    val = mat_lookup.get((p1, p2))
                    if val and val > best:
                        best = val
            cached = best
            next_step_cache[key] = cached
        return cached

    rows = []
    for entity1, entity2 in comembership_clique.iloc[:, [0, 1]].values:
        if entity1 in dico_most_precise_parent and entity2 in dico_most_precise_parent:
            parents1, ancestors_pathways1 = get_ancestors_for_entity(entity1)
            parents2, ancestors_pathways2 = get_ancestors_for_entity(entity2)

            mica = get_mica(parents1, parents2)
            score_IC_UP_mica = dict_score_ic_pathways[mica]

            max_next_step_pathway_score = get_max_next_step_score(
                ancestors_pathways1, ancestors_pathways2
            )

            score_comembership = max(score_IC_UP_mica, max_next_step_pathway_score)

            if score_comembership == score_IC_UP_mica:
                rows.append([entity1, entity2, score_comembership, "scoreMICA"])
            else:
                rows.append([entity1, entity2, score_comembership, "scoreNextStep"])

    weighted_comembership_clique = pd.DataFrame(
        rows, columns=["entity1", "entity2", "scoreComembership", "provenanceScore"]
    )
    weighted_comembership_clique = weighted_comembership_clique.sort_values(
        by="scoreComembership", ascending=False
    )
    weighted_comembership_clique.to_csv(
        f"../../Results/PathwayComembership/TGFbeta/R-HSA-170834_WeightedComembershipClique.csv",
        sep=",", header=True, index=False
    )

finally:
    process.kill()
    process.wait()

Pathway abstraction loaded in networkx: MultiDiGraph with 7 nodes and 12 edges


10:58:52 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Data/BioPAX/ReactomeBioPAX/R-HSA-170834.xml
10:58:52 INFO  Server          :: Dataset: in-memory: load file: /home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/../../Data/BioPAX/BioPAXOntology/biopax-level3.owl
10:58:52 INFO  Server          :: Running in read-only mode for /tgf_beta
10:58:52 INFO  Server          :: Apache Jena Fuseki 4.9.0
10:58:53 INFO  Config          :: FUSEKI_HOME=/home/cbeust/Softwares/JenaFuseki/apache-jena-fuseki-4.9.0
10:58:53 INFO  Config          :: FUSEKI_BASE=/home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/run
10:58:53 INFO  Config          :: Shiro file: file:///home/cbeust/Projects/2026/AbstractionReactomeTopPathways/Scripts/03_PathwayComembershipAbstraction/run/shiro.ini
10:58:53 INFO 

93


10:59:21 INFO  Fuseki          :: [1] GET http://localhost:3030/tgf_beta?query=%0APREFIX+rdf%3A+%3Chttp%3A//www.w3.org/1999/02/22-rdf-syntax-ns%23%3E%0APREFIX+rdfs%3A+%3Chttp%3A//www.w3.org/2000/01/rdf-schema%23%3E%0APREFIX+owl%3A+%3Chttp%3A//www.w3.org/2002/07/owl%23%3E%0APREFIX+bp3%3A+%3Chttp%3A//www.biopax.org/release/biopax-level3.owl%23%3E%0A%0ASELECT+%28COUNT%28DISTINCT+%3Fid%29+AS+%3FnbID%29%0AWHERE+%7B%0A++++VALUES+%3Fdb+%7B+%22UniProt%22+%22UniProt+Isoform%22+%7D%0A++++%3FentityRef+rdf%3Atype/%28rdfs%3AsubClassOf%2A%29+bp3%3AProteinReference+.%0A++++%3FentityRef+bp3%3Axref+%3FentityRefXref+.%0A++++%3FentityRefXref+bp3%3Adb+%3Fdb+.%0A++++%3FentityRefXref+bp3%3Aid+%3Fid+.%0A%7D%0A&format=json&output=json&results=json
10:59:21 INFO  Fuseki          :: [1] Query =  PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> PREFIX owl: <http://www.w3.org/2002/07/owl#> PREFIX bp3: <http://www.biopax.org/release/biopax-level3.owl#> 

In [11]:
print(weighted_comembership_clique.head())
fig = px.histogram(weighted_comembership_clique, x="scoreComembership", color="provenanceScore")
fig.show()

     entity1 entity2  scoreComembership provenanceScore
1175  P61586  Q9Y624           1.760011       scoreMICA
3083  Q05513  Q6ZSZ5           1.760011       scoreMICA
313   Q6ZSZ5  Q9NPB6           1.760011       scoreMICA
316   Q6ZSZ5  Q9P2M7           1.760011       scoreMICA
3096  Q05513  Q8TEW0           1.760011       scoreMICA
